# 분류와 평가지표

1. 분류란 무엇인가?
2. 여섯가지 알고리즘 비교
3. KNN
4. 앙상블 - 배깅과 부스팅
5. 평가지표

In [3]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _style import setup
from _ml import build_dataset, FEATURES, SEED, time_split, load_merged

setup()
pd.set_option("display.width", 130)

df = build_dataset()
train, test, cutoff = time_split(df)

tr = train.sample(15000, random_state=SEED)
te = test.sample(4000, random_state=SEED)

X_train, y_train = tr[FEATURES], tr["target_up"]
X_test, y_test = te[FEATURES], te["target_up"]

print(f" 학습 {len(tr):,}행 ({tr.date.min().date()} ~ {tr.date.max().date()})")
print(f" 테스트 {len(te):,}행 ({te.date.min().date()} ~ {te.date.max().date()})")
print(f" 상승 비율  학습 {y_train.mean():.4f} / 테스트 {y_test.mean():.4f}")

print("준비완료")

[폰트] Malgun Gothic / unicode_minus=False
 학습 15,000행 (2023-10-24 ~ 2026-01-16)
 테스트 4,000행 (2026-01-19 ~ 2026-08-07)
 상승 비율  학습 0.4931 / 테스트 0.4825
준비완료


---
# 1. 분류란 무엇인가?

**범주를 예측하는 문제**다.

| 종류 | 예시 |
| --- | --- |
| 이진분류 | 상승/하락, 스팸여부, 연체여부,... |
| 다중분류 | 섹터 예측, 신용등급(1, 2, 3, ..), ... |

우리는 이진분류를 중점적으로 다루겠다.
-> 지표가 명확하고 가장 많이 사용하는 형태라서

수익률, 거래량 비율, 이평대비, 변동성 -> up / down

> 오늘 장이 끝나기 전에 오늘 주식이 오를 것인가?를 예측한다.

---
# 2. 여섯가지 알고리즘

> 사용법 - 모든 알고리즘을 동일하게 사용할 수 있다.
> model = 어떤 모델()
> model.fit(X_train, y_train)
> pred = model.predict(X_test)
> score = model.score(X_test, y_test)

| 모델 | 스케일링 | 속도 | 해석 | 성능 |
| --- | --- | --- | --- | --- |
| LogisticRegression | 필요 | 빠름 | O | 보통 |
| DecisionTree | 불필요 | 빠름 | O | 낮음 |
| KNN | 필수 | 예측이 느림 | △ | 보통 |
| RandomForest | 불필요 | 보통 | △ | 좋음 |
| SVM | 필요 | 매우 느림 | X | 그때그때 다름 |
| GradientBoosting | 불필요 | 느림 | △ | 좋음 |

In [6]:
from sklearn.dummy import DummyClassifier   # 다른 모델 성능을 비교할 기준선을 만드는 분류기
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier     # 여러 개의 알고리즘이 합쳐진 분류모델
from sklearn.linear_model import LogisticRegression     # 선형 모델
from sklearn.metrics import accuracy_score, f1_score    # accuracy_score(정확도), f1_score(정밀도, 재현률의 평균)
from sklearn.neighbors import KNeighborsClassifier      # KNN모델
from sklearn.pipeline import Pipeline                   # 데이터 전처리부터 모델 학습 과정을 하나로 묶을 수 있게 도와주는 도구
from sklearn.preprocessing import StandardScaler        # 각 피처의 평균을 0, 표준편차를 1로 만들어 데이터 단위를 맞춰주는 전처리 도구
from sklearn.svm import SVC                             # SVM모델
from sklearn.tree import DecisionTreeClassifier         # 의사결정나무 tree모델


# 스케일링이 필요한 모델은 Pipeline으로 묶어준다.
def with_scaler(model):
    return Pipeline([("sc", StandardScaler()), ("m", model)])

# SVM은 데이터 수의 제곱에 비례해 느려짐. 그래서 표본을 줄여서 사용.
SVM_N = 3000

models = {
    "DummyClassifier": (DummyClassifier(strategy="most_frequent"), False),
    "LogisticRegression": (with_scaler(LogisticRegression(max_iter=1000, random_state=SEED)), False),
    "DecisionTree(제한)": (DecisionTreeClassifier(max_depth=5, random_state=SEED), False),
    "DecisionTree(제한없음)": (DecisionTreeClassifier(random_state=SEED), False),
    "KNN(k-50)": (with_scaler(KNeighborsClassifier(n_neighbors=50, n_jobs=-1)), False),
    "RandomForest": (RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED), False),
    "GradientBoosting": (GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED), False),
    "SVM(표본 3천개)": (with_scaler(SVC(random_state=SEED)), True)
}

rows = []
for name, (model, small) in models.items():
    # SVM만 작은 표본으로 진행
    Xtr = X_train.iloc[:SVM_N] if small else X_train
    ytr = y_train.iloc[:SVM_N] if small else y_train

    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    fit_time = time.perf_counter() - t0

    pred = model.predict(X_test)
    rows.append({
        "모델" : name,
        "학습" : model.score(Xtr, ytr),
        "테스트" : accuracy_score(y_test, pred),
        "F1" : f1_score(y_test, pred, zero_division=0),
        "학습시간" : fit_time,
    })

result = pd.DataFrame(rows)
print(result.round(4).to_string(index=False))

                모델     학습    테스트     F1   학습시간
   DummyClassifier 0.5069 0.5175 0.0000 0.0016
LogisticRegression 0.5107 0.4805 0.3859 0.0339
  DecisionTree(제한) 0.5219 0.4810 0.5245 0.0599
DecisionTree(제한없음) 1.0000 0.5005 0.4893 0.1515
         KNN(k-50) 0.5672 0.5178 0.4708 0.0154
      RandomForest 0.6105 0.4932 0.4269 0.9957
  GradientBoosting 0.5987 0.4960 0.4434 2.3369
       SVM(표본 3천개) 0.5720 0.4915 0.3787 0.1539
